In [1]:
import numpy as np
import random
import tensorflow as tf

# Set random seed for Python
random.seed(42)
# Set random seed for NumPy
np.random.seed(42)
# Set random seed for TensorFlow
tf.random.set_seed(42)

In [2]:
import os
# Get the name of the current conda environment
env_name = os.getenv("CONDA_DEFAULT_ENV")
print(f"The current Conda environment is: {env_name}")

The current Conda environment is: tensorflowgpu


In [3]:
# import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# import matplotlib.pyplot as plt
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.layers import Layer, Multiply, Dense, Input
from tensorflow.keras.models import Model
from scipy.stats import pearsonr
import tensorflow as tf

# Check if TensorFlow can access a GPU
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [4]:
import numpy
print(numpy.__version__)

1.26.4


In [5]:
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.attention_dense = None  # Will be initialized in build()

    def build(self, input_shape):
        self.attention_dense = Dense(input_shape[-1], activation='softmax')
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.attention_dense(inputs)
        weighted_output = Multiply()([inputs, attention_weights])
        return weighted_output

    def compute_output_shape(self, input_shape):
        return input_shape

In [6]:
def nse(y_true, y_pred):
    return 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
import numpy as np

def pbias(y_true, y_pred):
    pbias_value = 100 * np.sum(y_true - y_pred) / np.sum(y_true)
    return float(pbias_value)


def kge(y_true, y_pred):
    # Calculate the Pearson correlation coefficient (r)
    r, _ = pearsonr(y_true, y_pred)
    
    # Calculate the mean of the observed and predicted values
    mu_true = np.mean(y_true)
    mu_pred = np.mean(y_pred)
    
    # Calculate the standard deviation of the observed and predicted values
    sigma_true = np.std(y_true)
    sigma_pred = np.std(y_pred)
    
    # Compute the KGE
    kge_value = 1 - np.sqrt((r - 1)**2 + (sigma_pred / sigma_true - 1)**2 + (mu_pred / mu_true - 1)**2)
    
    return kge_value


In [7]:
# Define start and end dates as datetime objects
start_date = pd.to_datetime("2000-01-01")
end_date = pd.to_datetime("2100-12-31")

In [8]:
epochs=256
batch_size=16
os.chdir('F:/geodata/river_runoff_obs')
GCM_name = 'canesm5'
[ 'canesm5',
 'cnrm-cm6-1',
 'cnrm-esm2-1',
 'ec-earth3',
 'gfdl-esm4',
 'ipsl-cm6a-lr',
 'miroc6',
 'mpi-esm1-2-hr',
 'mri-esm2-0',
 'ukesm1-0-ll'
]
# This 2 GCMs can only be obtained in r1i1p2

['canesm5',
 'cnrm-cm6-1',
 'cnrm-esm2-1',
 'ec-earth3',
 'gfdl-esm4',
 'ipsl-cm6a-lr',
 'miroc6',
 'mpi-esm1-2-hr',
 'mri-esm2-0',
 'ukesm1-0-ll']

In [9]:
valid_rate = 0.3

In [10]:
# from datetime import datetime# Get current time
# os.chdir(r'H:\new_folder2')
# current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# for i in range(1,31):
#
#     txtbook_path = f'output_data\\note_{current_time}_mon.txt'
#     print(txtbook_path)
#     for GCM_name in [
#         'canesm5',
#      'cnrm-cm6-1',
#      'cnrm-esm2-1',
#      'ec-earth3',
#      'gfdl-esm4',
#      'ipsl-cm6a-lr',
#      'miroc6',
#      'mpi-esm1-2-hr',
#      'mri-esm2-0',
#      'ukesm1-0-ll'
#     ]:
#     # This 2 GCMs can only be obtained in r1i1p2]:
#         print(GCM_name,epochs,batch_size,valid_rate)
#         print(epochs,batch_size,valid_rate)
#         for scenario in ['ssp126','ssp370','ssp585']:
#             print(scenario)
#
#             # Dataset loading
#             csv_path = f'input_data\\{i}_{GCM_name}_{scenario}_future.csv'
#             usecols = ['date', 'prcp', 'tmp','glacier_runoff','runoff','ET']
#
#             df_full = pd.read_csv(csv_path, usecols=usecols)
#             # df_full.columns = ['date', 'prcp', 'tmp','runoff','ET','glacier_runoff']
#             # df_full = df_full[-1212:]
#             df = df_full.dropna()
#             df_future = df_full[df_full['runoff'].isnull()]
#             file_name = f'{i}_{GCM_name}_r1i1p1f1_{scenario}_{epochs}_{batch_size}'
#             # Prepare features (X) and targets (y)
#             X = df[['prcp', 'tmp','glacier_runoff']].values  # Inputs: precipitation, temperature, mass balance
#             y = df[['runoff','ET']].values         # Outputs: runoff (dis) and evaporation (ET)
#
#             # Split the data into training and testing sets
#             X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=valid_rate, random_state=42)
#
#             # Standardize the data
#             scaler_X = StandardScaler()
#             scaler_y = StandardScaler()
#
#             X_train = scaler_X.fit_transform(X_train)
#             X_test = scaler_X.transform(X_test)
#             y_train = scaler_y.fit_transform(y_train)
#             y_test = scaler_y.transform(y_test)
#
#
#
#             # Define the model
#             inputs = Input(shape=(X_train.shape[1],))  # Input layer
#             x = Dense(64, activation='relu')(inputs)  # Hidden layer 1
#             x = Dense(32, activation='relu')(x)       # Hidden layer 2
#             x = AttentionLayer()(x)                   # Attention layer
#             x = Dense(2)(x)                           # Output layer (2 units for runoff and evaporation)
#
#             # Create the model
#             model = Model(inputs, x)
#
#             # Add Dropout after defining the layers
#             # model.add(Dropout(0.2))  # Apply a dropout rate of 20%
#
#             # Compile the model
#             model.compile(optimizer='adam', loss='mse')
#
#             # Print the model summary
#             model.summary()
#
#             # Train the model
#             model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=valid_rate, )
#
#             # Make predictions
#             predictions = model.predict(X_test)
#             predictions_rescaled = scaler_y.inverse_transform(predictions)
#
#             # Convert predictions to DataFrame for runoff and evaporation
#             pred_df = pd.DataFrame(predictions_rescaled, columns=['predicted_runoff', 'predicted_ET'])
#             print(pred_df.head())
#             from sklearn.metrics import mean_squared_error
#             # Calculate predictions and rescale
#             predictions = model.predict(X_test)
#             predictions_rescaled = scaler_y.inverse_transform(predictions)
#
#             # Separate the predicted and actual values for runoff and evaporation
#             y_test_rescaled = scaler_y.inverse_transform(y_test)
#             runoff_observed = y_test_rescaled[:, 0]
#             evaporation_observed = y_test_rescaled[:, 1]
#             runoff_predicted = predictions_rescaled[:, 0]
#             evaporation_predicted = predictions_rescaled[:, 1]
#
#
#             # Assuming future_df is loaded and has the same columns as df
#             # Extract features from future_df
#             X_full = df_full[['prcp', 'tmp','glacier_runoff']].values  # Only the input features
#             # Standardize features based on the training data scaler
#             X_full_scaled = scaler_X.transform(X_full)
#
#             # Convert the 'time' column to datetime format if needed
#             df.loc[:,'date'] = pd.to_datetime(df['date'])
#             df_full.loc[:,'date'] = pd.to_datetime(df_full['date'])
#             # Ensure 'date' is set as the index for both DataFrames if not already
#             df.set_index('date', inplace=True)
#             df_full.set_index('date', inplace=True)
#
#
#             # Predict future runoff and evaporation
#             full_predictions_scaled = model.predict(X_full_scaled)
#             full_predictions_scaled = model.predict(X_full_scaled)
#
#             # Rescale predictions to original scale
#             full_predictions = scaler_y.inverse_transform(full_predictions_scaled)
#
#             # Convert predictions to DataFrame for readability
#             df_full[['Projected_Runoff', 'Projected_Evaporation']] = full_predictions
#             print(df_full[['Projected_Runoff', 'Projected_Evaporation']].head())
#
#
#
#             historic_pred_df = df_full.dropna(subset='runoff')
#             historic_pred_df = historic_pred_df.tail(int(valid_rate * len(historic_pred_df)))
#             # Define the NSE function
#
#             # Calculate NSE for runoff and evaporation
#             nse_runoff = nse(historic_pred_df.runoff, historic_pred_df.Projected_Runoff)
#             nse_evaporation = nse(historic_pred_df.ET, historic_pred_df.Projected_Evaporation)
#             kge_runoff = kge(historic_pred_df.runoff, historic_pred_df.Projected_Runoff)
#             pbias_runoff = pbias(historic_pred_df.runoff, historic_pred_df.Projected_Runoff)
#             print('nse_runoff:',nse_runoff,'nse_ET:',nse_evaporation,'kge_runoff:',kge_runoff,'pbias_runoff:',pbias_runoff)
#
#
#             # # Plot the line chart for Projected_Runoff
#             # plt.plot(historic_pred_df.index, historic_pred_df['Projected_Runoff'], label='Projected Runoff', color='red')
#             #
#             # # Plot the scatter plot for runoff
#             # plt.scatter(historic_pred_df.index, historic_pred_df['runoff'], label='Observed Runoff', color='b')
#             #
#             # # Add labels, title, and legend
#             # plt.xlabel('Index')
#             # plt.ylabel(r'Runoff ($\mathrm{m^3 \cdot s^{-1}}$)')  # Use LaTeX for units
#             # plt.title('Projected Runoff vs Observed Runoff')
#             # plt.xlim(([pd.to_datetime("2014-01"), pd.to_datetime("2020-01-01")]))
#             # plt.legend()
#             #
#             # # Show the plot
#             # plt.show()
#             #
#             # # Plot with a larger figure size
#             # ax = df_full[[ 'Projected_Runoff','runoff']].plot(figsize=(12, 6))
#             # # Optional: Rotate x-tick labels for better readability
#             # plt.xticks(rotation=45)
#             # plt.title(f"Historical and projected runoff of {file_name}")
#             # # Show the plot
#             # plt.show()
#             # # Plot with a larger figure size
#             # ax = df_full[['ET', 'Projected_Evaporation']].plot(figsize=(12, 6))
#             # # Optional: Rotate x-tick labels for better readability
#             # plt.xticks(rotation=45)
#             # plt.title(f"Historical and projected evaporation of {file_name}")
#             # # Show the plot
#             # plt.show()
#             #
#             # # Generate a range of ticks every 20 years
#             # x_ticks = pd.date_range(start=start_date, end=end_date, freq='10Y')
#             #
#             # # Set the size of the figure
#             # fig, axes = plt.subplots(5, 1, figsize=(16, 9), sharex=True,facecolor='w')
#             # title_list = ['Monthly Precipitation','Monthly Temperature','Monthly Glacier Runoff','Monthly Runoff','Monthly Evaporation']
#             #
#             # # Define y-axis limits for each plot
#             # y_lims = [(0, 70), (-20, 25), (0, 3.3* 1e9), (0, 1200), (0, 800)]
#             #
#             # # Loop through the columns and set y-axis limits
#             # for i, col in enumerate(['pre', 'tm','gr',  ['runoff', 'Projected_Runoff'], ['ET', 'Projected_Evaporation']]):
#             #     df_full[col].plot(ax=axes[i])
#             #
#             #     # # Set y-axis limits
#             #     # axes[i].set_ylim(y_lims[i])
#             #
#             #     # Optional: Add y-axis label, legend, title, grid, etc.
#             #     # axes[i].set_ylabel(col)  # Set y-axis label to the column name
#             #     axes[i].legend(loc='upper right')  # Optional: add legend
#             #     axes[i].set_title(title_list[i])
#             #
#             #     # Set x-axis limits
#             #     axes[i].set_xlim([start_date, end_date])
#             #     axes[i].set_xticks(x_ticks)
#             #
#             #     # Set x-axis labels as years (2000, 2020, ..., 2100)
#             #     axes[i].set_xticklabels([str(date.year) for date in x_ticks], rotation=0)
#             #
#             #     axes[i].grid(True)  # Optional: add grid for clarity
#             #
#             #
#             #
#             #
#             # # Set x-axis label for the entire figure
#             # axes[-1].set_xlabel('Date')  # or adjust label based on your x-axis
#             #
#             # # Adjust layout to prevent overlap
#             # plt.tight_layout()
#             # plt.savefig(f'{file_name}.svg')
#             # plt.show()
#
#             df_full.to_csv(f"output_data\\{file_name}_project.csv")
#             # annual_df = df_full.resample('Y').agg({
#             #     'pre': 'sum',  # If there are any NaNs in the group, the sum will be NaN
#             #     'tm': 'mean',  # The mean will also return NaN if there are NaNs in the group
#             #     'gr': 'sum',   # Same for sum, will return NaN if any NaNs are present
#             #     'runoff': 'sum',
#             #     'ET': 'sum',
#             #     'Projected_Runoff': 'sum',
#             #     'Projected_Evaporation': 'sum'
#             # }, skipna=False)  # Ensures NaNs are preserved in aggregation
#             #
#             # # Reset index if needed
#             # annual_df.reset_index(inplace=True)
#             # annual_df[['ET', 'runoff', 'Projected_Runoff',  'Projected_Evaporation']]=annual_df[['ET', 'runoff', 'Projected_Runoff',  'Projected_Evaporation']].replace(0, np.nan,)
#             #
#             # # Display the result
#             # print(annual_df.head())
#
#             # --- Prepare input sample ---
#             # Select one instance to explain (e.g., the first sample in test set)
#             input_index = 0
#             X_sample = X_test[input_index]  # already standardized
#             baseline = np.zeros_like(X_sample)  # baseline (zero vector)
#             baseline = baseline.astype(np.float32)
#             X_sample = X_sample.astype(np.float32)
#
#             # append the information at the end of text book
#             with open(txtbook_path, 'a') as file:
#                 file.write(f"Name: {GCM_name}_, Scenario: {scenario}, Epochs: {epochs}, Batch Size: {batch_size}, Valid Rate: {valid_rate},NSE Runoff: {nse_runoff}, KGE Runoff: {kge_runoff},Pbias Runoff: {pbias_runoff}\n")
#
#
#
# import io
# with open(txtbook_path, 'a') as file:
#     # Capture the summary
#     summary_io = io.StringIO()
#     model.summary(print_fn=lambda x: summary_io.write(x + "\n"))
#     model_summary = summary_io.getvalue()
#     file.write(model_summary+'\n')

In [ ]:
from datetime import datetime# Get current time
os.chdir(r'J:\new_folder2')
current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
for i in range(1,31):
    # if i != 28:
    #     continue
    # if i in [4,6]:
    #     continue
    txtbook_path = f'output_data\\note_{current_time}_mon.txt'
    print(txtbook_path)
    for GCM_name in [
        'canesm5',
     'cnrm-cm6-1',
     'cnrm-esm2-1',
     'ec-earth3',
     'gfdl-esm4',
     'ipsl-cm6a-lr',
     'miroc6',
     'mpi-esm1-2-hr',
     'mri-esm2-0',
     'ukesm1-0-ll'
    ]:
    # This 2 GCMs can only be obtained in r1i1p2]:
        print(GCM_name,epochs,batch_size,valid_rate)
        print(epochs,batch_size,valid_rate)
        for scenario in ['ssp126','ssp370','ssp585']:
            print(scenario)

            # Dataset loading
            csv_path = f'input_data\\{i}_{GCM_name}_{scenario}_future.csv'
            usecols = ['date', 'prcp', 'tmp','glacier_runoff','runoff','ET']

            df_full = pd.read_csv(csv_path, usecols=usecols)
            # df_full.columns = ['date', 'prcp', 'tmp','runoff','ET','glacier_runoff']
            # df_full = df_full[-1212:]
            df = df_full.dropna()
            df_future = df_full[df_full['runoff'].isnull()]
            file_name = f'{i}_{GCM_name}_r1i1p1f1_{scenario}_{epochs}_{batch_size}'
            # Prepare features (X) and targets (y)
            X = df[['prcp', 'tmp','glacier_runoff']].values  # Inputs: precipitation, temperature, mass balance
            y = df[['runoff','ET']].values         # Outputs: runoff (dis) and evaporation (ET)

            # Split the data into training and testing sets
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=valid_rate, random_state=42)

            # Standardize the data
            scaler_X = StandardScaler()
            scaler_y = StandardScaler()

            X_train = scaler_X.fit_transform(X_train)
            X_test = scaler_X.transform(X_test)
            y_train = scaler_y.fit_transform(y_train)
            y_test = scaler_y.transform(y_test)

            # Input layer
            inputs = Input(shape=(X_train.shape[1],))

            # Hidden layers
            x = Dense(64, activation='relu')(inputs)
            x = Dense(32, activation='relu')(x)

            # Attention layer (assuming you’ve defined AttentionLayer elsewhere)
            x = AttentionLayer()(x)

            # Output layer: 2 units with 'relu' to ensure non-negative output
            x = Dense(2, activation='relu')(x)

            # Optional: Add Dropout after dense layers for regularization
            # If desired, you could place it after the second Dense layer
            # x = Dropout(0.2)(x)

            # Build model
            model = Model(inputs, x)

            # Compile model
            model.compile(optimizer='adam', loss='mse')

            # Print the model summary
            model.summary()

            # Train the model
            model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=valid_rate, )

            # Make predictions
            predictions = model.predict(X_test)
            predictions_rescaled = scaler_y.inverse_transform(predictions)

            # Convert predictions to DataFrame for runoff and evaporation
            pred_df = pd.DataFrame(predictions_rescaled, columns=['predicted_runoff', 'predicted_ET'])
            print(pred_df.head())
            from sklearn.metrics import mean_squared_error
            # Calculate predictions and rescale
            predictions = model.predict(X_test)
            predictions_rescaled = scaler_y.inverse_transform(predictions)

            # Separate the predicted and actual values for runoff and evaporation
            y_test_rescaled = scaler_y.inverse_transform(y_test)
            runoff_observed = y_test_rescaled[:, 0]
            evaporation_observed = y_test_rescaled[:, 1]
            runoff_predicted = predictions_rescaled[:, 0]
            evaporation_predicted = predictions_rescaled[:, 1]


            # Assuming future_df is loaded and has the same columns as df
            # Extract features from future_df
            X_full = df_full[['prcp', 'tmp','glacier_runoff']].values  # Only the input features
            # Standardize features based on the training data scaler
            X_full_scaled = scaler_X.transform(X_full)

            # Convert the 'time' column to datetime format if needed
            df.loc[:,'date'] = pd.to_datetime(df['date'])
            df_full.loc[:,'date'] = pd.to_datetime(df_full['date'])
            # Ensure 'date' is set as the index for both DataFrames if not already
            df.set_index('date', inplace=True)
            df_full.set_index('date', inplace=True)


            # Predict future runoff and evaporation
            full_predictions_scaled = model.predict(X_full_scaled)
            full_predictions_scaled = model.predict(X_full_scaled)

            # Rescale predictions to original scale
            full_predictions = scaler_y.inverse_transform(full_predictions_scaled)

            # Convert predictions to DataFrame for readability
            df_full[['Projected_Runoff', 'Projected_Evaporation']] = full_predictions
            print(df_full[['Projected_Runoff', 'Projected_Evaporation']].head())



            historic_pred_df = df_full.dropna(subset='runoff')
            historic_pred_df = historic_pred_df.tail(int(valid_rate * len(historic_pred_df)))
            # Define the NSE function

            # Calculate NSE for runoff and evaporation
            nse_runoff = nse(historic_pred_df.runoff, historic_pred_df.Projected_Runoff)
            nse_evaporation = nse(historic_pred_df.ET, historic_pred_df.Projected_Evaporation)
            kge_runoff = kge(historic_pred_df.runoff, historic_pred_df.Projected_Runoff)
            pbias_runoff = pbias(historic_pred_df.runoff, historic_pred_df.Projected_Runoff)
            print('nse_runoff:',nse_runoff,'nse_ET:',nse_evaporation,'kge_runoff:',kge_runoff,'pbias_runoff:',pbias_runoff)


            # # Plot the line chart for Projected_Runoff
            # plt.plot(historic_pred_df.index, historic_pred_df['Projected_Runoff'], label='Projected Runoff', color='red')
            #
            # # Plot the scatter plot for runoff
            # plt.scatter(historic_pred_df.index, historic_pred_df['runoff'], label='Observed Runoff', color='b')
            #
            # # Add labels, title, and legend
            # plt.xlabel('Index')
            # plt.ylabel(r'Runoff ($\mathrm{m^3 \cdot s^{-1}}$)')  # Use LaTeX for units
            # plt.title('Projected Runoff vs Observed Runoff')
            # plt.xlim(([pd.to_datetime("2014-01"), pd.to_datetime("2020-01-01")]))
            # plt.legend()
            #
            # # Show the plot
            # plt.show()
            #
            # # Plot with a larger figure size
            # ax = df_full[[ 'Projected_Runoff','runoff']].plot(figsize=(12, 6))
            # # Optional: Rotate x-tick labels for better readability
            # plt.xticks(rotation=45)
            # plt.title(f"Historical and projected runoff of {file_name}")
            # # Show the plot
            # plt.show()
            # # Plot with a larger figure size
            # ax = df_full[['ET', 'Projected_Evaporation']].plot(figsize=(12, 6))
            # # Optional: Rotate x-tick labels for better readability
            # plt.xticks(rotation=45)
            # plt.title(f"Historical and projected evaporation of {file_name}")
            # # Show the plot
            # plt.show()
            #
            # # Generate a range of ticks every 20 years
            # x_ticks = pd.date_range(start=start_date, end=end_date, freq='10Y')
            #
            # # Set the size of the figure
            # fig, axes = plt.subplots(5, 1, figsize=(16, 9), sharex=True,facecolor='w')
            # title_list = ['Monthly Precipitation','Monthly Temperature','Monthly Glacier Runoff','Monthly Runoff','Monthly Evaporation']
            #
            # # Define y-axis limits for each plot
            # y_lims = [(0, 70), (-20, 25), (0, 3.3* 1e9), (0, 1200), (0, 800)]
            #
            # # Loop through the columns and set y-axis limits
            # for i, col in enumerate(['pre', 'tm','gr',  ['runoff', 'Projected_Runoff'], ['ET', 'Projected_Evaporation']]):
            #     df_full[col].plot(ax=axes[i])
            #
            #     # # Set y-axis limits
            #     # axes[i].set_ylim(y_lims[i])
            #
            #     # Optional: Add y-axis label, legend, title, grid, etc.
            #     # axes[i].set_ylabel(col)  # Set y-axis label to the column name
            #     axes[i].legend(loc='upper right')  # Optional: add legend
            #     axes[i].set_title(title_list[i])
            #
            #     # Set x-axis limits
            #     axes[i].set_xlim([start_date, end_date])
            #     axes[i].set_xticks(x_ticks)
            #
            #     # Set x-axis labels as years (2000, 2020, ..., 2100)
            #     axes[i].set_xticklabels([str(date.year) for date in x_ticks], rotation=0)
            #
            #     axes[i].grid(True)  # Optional: add grid for clarity
            #
            #
            #
            #
            # # Set x-axis label for the entire figure
            # axes[-1].set_xlabel('Date')  # or adjust label based on your x-axis
            #
            # # Adjust layout to prevent overlap
            # plt.tight_layout()
            # plt.savefig(f'{file_name}.svg')
            # plt.show()

            df_full.to_csv(f"output_data\\{file_name}_project.csv")
            # annual_df = df_full.resample('Y').agg({
            #     'pre': 'sum',  # If there are any NaNs in the group, the sum will be NaN
            #     'tm': 'mean',  # The mean will also return NaN if there are NaNs in the group
            #     'gr': 'sum',   # Same for sum, will return NaN if any NaNs are present
            #     'runoff': 'sum',
            #     'ET': 'sum',
            #     'Projected_Runoff': 'sum',
            #     'Projected_Evaporation': 'sum'
            # }, skipna=False)  # Ensures NaNs are preserved in aggregation
            #
            # # Reset index if needed
            # annual_df.reset_index(inplace=True)
            # annual_df[['ET', 'runoff', 'Projected_Runoff',  'Projected_Evaporation']]=annual_df[['ET', 'runoff', 'Projected_Runoff',  'Projected_Evaporation']].replace(0, np.nan,)
            #
            # # Display the result
            # print(annual_df.head())

            # --- Prepare input sample ---
            # Select one instance to explain (e.g., the first sample in test set)
            input_index = 0
            X_sample = X_test[input_index]  # already standardized
            baseline = np.zeros_like(X_sample)  # baseline (zero vector)
            baseline = baseline.astype(np.float32)
            X_sample = X_sample.astype(np.float32)

            # append the information at the end of text book
            with open(txtbook_path, 'a') as file:
                file.write(f"Name: {GCM_name}_, Scenario: {scenario}, Epochs: {epochs}, Batch Size: {batch_size}, Valid Rate: {valid_rate},NSE Runoff: {nse_runoff}, KGE Runoff: {kge_runoff},Pbias Runoff: {pbias_runoff}\n")



import io
with open(txtbook_path, 'a') as file:
    # Capture the summary
    summary_io = io.StringIO()
    model.summary(print_fn=lambda x: summary_io.write(x + "\n"))
    model_summary = summary_io.getvalue()
    file.write(model_summary+'\n')

output_data\note_2025-07-23_19-19-06_mon.txt
canesm5 256 16 0.3
256 16 0.3
ssp126
Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 3)]               0         
                                                                 
 dense_9 (Dense)             (None, 64)                256       
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 attention_layer_3 (Attentio  (None, 32)               1056      
 nLayer)                                                         
                                                                 
 dense_11 (Dense)            (None, 2)                 66        
                                                                 
Total params: 3,458
Trainable params: 3,458

In [17]:
df_full

,prcp,tmp,ET,runoff,glacier_runoff,Projected_Runoff,Projected_Evaporation
date,,,,,,,
1901-01-01,25.841267,255.565674,NaN,NaN,NaN,NaN,NaN
1901-02-01,15.906990,260.885803,NaN,NaN,NaN,NaN,NaN
1901-03-01,12.276011,269.737518,NaN,NaN,NaN,NaN,NaN
1901-04-01,14.595119,281.450073,NaN,NaN,NaN,NaN,NaN
1901-05-01,19.266657,288.621155,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
2100-08-01,59.507439,293.343323,NaN,NaN,2.819832e+08,392.643799,50.521446
2100-09-01,34.085693,287.787598,NaN,NaN,5.084546e+07,227.960892,37.758999
2100-10-01,11.081620,281.230804,NaN,NaN,4.417463e+05,67.501213,13.116805
